# USD calibration — results

Reads the latest fit in `dm_calibration/runs/` and shows it: error metrics, model-vs-market ATM term structure + smiles, the 3-D overlay, and the loss curve. Re-run any cell after a new calibration to refresh. (No GPU needed — just reads the result CSVs.)

In [ ]:
import os, sys, json
cand = os.path.abspath('')
DM = None
while cand != os.path.dirname(cand):
    if os.path.exists(os.path.join(cand, 'dm_calibrate.py')): DM = cand; break
    if os.path.exists(os.path.join(cand, 'dm_calibration', 'dm_calibrate.py')): DM = os.path.join(cand, 'dm_calibration'); break
    cand = os.path.dirname(cand)
sys.path.insert(0, DM)
import pandas as pd, numpy as np, dm_figures as dfig
from IPython.display import Image, display
CCY, MODEL = 'usd', 'HWH_qe'
RUNS = os.path.join(DM, 'runs')
res = pd.read_csv(os.path.join(RUNS, f'results_{CCY}_{MODEL}.csv')); res['strike'] = res.strike.astype(float)
print(f'{CCY.upper()} {MODEL}: {len(res)} caplets')

## Nested-model ladder — Roland rec 2 (all four models)
The headline comparison: `1F_Gauss -> 2F_Gauss -> HW_detvol -> HWH` on the identical surface — each rung's RMSE gap quantifies one modelling ingredient (2nd factor, vol term structure, stochastic vol). Populates as the ladder run completes; re-run the notebook to refresh.

In [ ]:
lf = dfig.ladder_frame(CCY)
print(f'ladder rungs available: {len(lf)} of 4')
display(lf)
display(Image(dfig.fig_ladder(CCY)))

## Error metrics (Roland rec 4: overall / relative / maturity buckets / ATM-OTM)

In [ ]:
print(json.dumps(dfig.error_metrics(res), indent=2))

## ATM term structure by maturity — market vs model

In [ ]:
atm = res[(res.strike - res.F).abs() < 0.003]
tab = atm.groupby(atm.t_fix.round(1)).agg(F_pct=('F', lambda x: 100*x.mean()),
        market_bp=('market_vol_bp', 'mean'), model_bp=('model_vol_bp', 'mean'))
tab['diff_bp'] = tab.model_bp - tab.market_bp
tab.round(1)

## Fit: ATM term structure + smile slices (market — solid, model - - dashed)

In [ ]:
display(Image(dfig.fig_fit(res, CCY)))

## 3-D overlay: market surface (mesh) vs model (surface)

In [ ]:
display(Image(dfig.fig_overlay(res, CCY)))

## Calibrated vs market — signum 3D overlay (patched)
**Market (blue) + calibrated model (orange) on one smooth surface.** signum's `Surface3D` is now patched: `dataShape` fixes the striping (echarts-gl was mis-detecting the non-square grid), it accepts multiple `.surface()` calls to overlay, and it renders via `srcdoc` in VSCode. Strike on a uniform index (real strikes printed above). Falls back to matplotlib if signum is unavailable.

In [ ]:
# signum 3D overlay - market + model on one smooth chart (patched: dataShape + multi-surface + srcdoc)
import numpy as np
pm = res.pivot_table(index='strike', columns='t_fix', values='market_vol_bp')
po = res.pivot_table(index='strike', columns='t_fix', values='model_vol_bp')
mats = pm.columns.values.astype(float); strikes = pm.index.values.astype(float) * 100
yidx = np.arange(len(strikes))
ylab = f'strike idx (0={strikes[0]:.1f}% .. {len(strikes)-1}={strikes[-1]:.1f}%)'
print('strike index -> %:', {int(i): round(float(s), 1) for i, s in enumerate(strikes)})
try:
    from signum.engine.surface3d import Surface3D
    (Surface3D(theme='midnight', height=620,
               title='EUR caplet IV - market (blue) vs calibrated model (orange), RMSE 15.2bp')
        .surface(x=mats, y=yidx, z=pm.values, name='market', color='#4A90D9', opacity=0.85,
                 x_label='TTM (Y)', y_label=ylab, z_label='vol bp')
        .surface(x=mats, y=yidx, z=po.values, name='model',  color='#E8843C', opacity=0.60)
        .show())
except Exception as e:
    print('signum unavailable, matplotlib fallback:', e)
    import matplotlib.pyplot as plt
    X, Yi = np.meshgrid(mats, yidx)
    fig = plt.figure(figsize=(11, 6)); ax = fig.add_subplot(111, projection='3d')
    ax.plot_surface(X, Yi, po.values, cmap='viridis', alpha=.85)
    ax.plot_wireframe(X, Yi, pm.values, color='k', lw=.4, alpha=.5)
    ax.set_xlabel('TTM (Y)'); ax.set_ylabel('strike idx'); ax.set_zlabel('vol bp'); ax.view_init(25, -60); plt.show()

## Smile slices — curvature, ATM dip, skew, high-strike wall
The 3-D surface hides shape; these 2-D slices are the real diagnostic. **Blue = market, orange = model**, dotted line = forward. Look for: the **dip** near the forward (curvature), the **skew** (asymmetry), and how far the model reaches into the **high-strike wall**.

In [ ]:
# Smile slices at selected maturities - market (blue) vs model (orange)
import numpy as np, matplotlib.pyplot as plt
mats = [1.0, 2.0, 5.0, 10.0]
fig, axs = plt.subplots(1, len(mats), figsize=(16, 3.6))
for ax, T in zip(axs, mats):
    s = res[np.isclose(res.t_fix, T)].sort_values('strike')
    if not len(s):
        continue
    ax.plot(s.strike * 100, s.market_vol_bp, 'o-', color='tab:blue', label='market')
    ax.plot(s.strike * 100, s.model_vol_bp, 's--', color='tab:orange', label='model')
    f = float(s.F.iloc[0]) * 100
    ax.axvline(f, color='grey', ls=':', lw=1)
    ax.set_title(f'{T:.0f}Y  (F~{f:.2f}%)'); ax.set_xlabel('strike %'); ax.grid(alpha=.3)
axs[0].set_ylabel('normal vol (bp)'); axs[0].legend()
plt.tight_layout(); plt.show()

## Calibration loss vs SPSA iteration

In [ ]:
hp = os.path.join(RUNS, f'history_{CCY}_{MODEL}.csv')
display(Image(dfig.fig_loss(CCY, MODEL))) if os.path.exists(hp) else print('no history csv yet')

---
**Read-out:** the long/mid end (≈5–30Y) fits well and the smile *shape* is right; the short end (0–3Y) is the open residual — same short-maturity difficulty as the RU thesis. See `article/article_plan.md` for the tuning experiments (QE simulation, lower vol-of-vol start, scheduled jumps).